# Write star-sited monomer SMILES

PolyzyMD builds dynamic polymer chains from **star-sited SMILES**. A monomer is
given as its repeat unit, as it is in the chain, with `*` on the two atoms that
bond to the neighbouring repeat units. A cap (end group) has one `*`.

This notebook helps you write them. Paste the SMILES of your monomer (for
example from PubChem), find the two linking atoms by their index, and copy the
star-sited SMILES into your config. Nothing here changes your config; check
each result by eye.

Run it in the PolyzyMD build environment:

```bash
pixi run -e build jupyter lab docs/source/tutorials/star_sited_monomers.ipynb
```

## 1. Draw your monomer

Paste the SMILES of the monomer. The default is sulfobetaine methacrylate
(SBMA, PubChem CID 79101), one of the two monomers of the SBMA-EGMA chains earlier
PolyzyMD releases built. Every
atom is labelled with its index.

In [ ]:
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D


def show(mol, highlight=(), size=(520, 320)):
    """Draw a molecule with every atom labelled by its index."""
    drawer = rdMolDraw2D.MolDraw2DSVG(*size)
    options = drawer.drawOptions()
    options.addAtomIndices = True
    options.annotationFontScale = 0.8
    drawer.DrawMolecule(mol, highlightAtoms=list(highlight))
    drawer.FinishDrawing()
    svg = drawer.GetDrawingText()
    try:
        from IPython.display import SVG, display
    except ImportError:  # outside Jupyter: nothing to draw on
        return
    display(SVG(svg))


monomer_smiles = "CC(=C)C(=O)OCC[N+](C)(C)CCCS(=O)(=O)[O-]"  # SBMA; paste yours here

monomer = Chem.MolFromSmiles(monomer_smiles)
assert monomer is not None, f"RDKit cannot read {monomer_smiles!r}"
show(monomer)

## 2. Find the two linking atoms

In the chain, each repeat unit is bonded to the one before it and the one after
it. The two atoms that make those bonds carry the `*`.

- For a vinyl, acrylate or methacrylate monomer, they are the two carbons of
  the C=C double bond. Polymerization opens that bond: in the chain it is a
  single bond, and each of its carbons gains one bond to a neighbouring unit.
- The **first** `*` bonds toward the head of the chain, the second toward the
  tail. For methacrylates, PolyzyMD's examples put the first `*` on the CH2
  carbon, so the chain grows head to tail.

The cell below lists the C=C double bonds outside rings and carbonyls, the
usual polymerizable bond. It does not choose for you.

In [ ]:
candidates = []
for bond in monomer.GetBonds():
    a, b = bond.GetBeginAtom(), bond.GetEndAtom()
    if (
        bond.GetBondType() == Chem.BondType.DOUBLE
        and a.GetSymbol() == "C"
        and b.GetSymbol() == "C"
        and not bond.IsInRing()
    ):
        candidates.append((a.GetIdx(), b.GetIdx()))
        print(f"C=C between atoms {a.GetIdx()} and {b.GetIdx()}")
if not candidates:
    print("No C=C outside a ring: find the linking atoms from the chemistry of your polymer.")
show(monomer, highlight=[i for pair in candidates for i in pair])

## 3. Place the stars

Give the indices of the two linking atoms, head first. The helper turns a
double bond between them into a single bond, puts `*` on each, and redraws the
repeat unit with the two linking atoms highlighted. PolyzyMD then reads the
result the way a build will.

In [ ]:
# The two linking atoms, head first. By default: the C=C listed above, its CH2
# carbon first. Check them against the drawing and change them if needed.
if candidates:
    a, b = candidates[0]
    hydrogens = monomer.GetAtomWithIdx(a).GetTotalNumHs(), monomer.GetAtomWithIdx(b).GetTotalNumHs()
    head_atom, tail_atom = (a, b) if hydrogens[0] >= hydrogens[1] else (b, a)
else:
    head_atom, tail_atom = 0, 1  # set these to your linking atoms
print(f"head atom {head_atom}, tail atom {tail_atom}")


def star_sited(mol, head, tail):
    """Return the repeat unit of ``mol`` with '*' on atoms ``head`` and ``tail``."""
    for atom in (head, tail):
        if atom >= mol.GetNumAtoms():
            raise ValueError(f"atom {atom} does not exist; the molecule has atoms 0 to {mol.GetNumAtoms() - 1}")
    editable = Chem.RWMol(mol)
    bond = editable.GetBondBetweenAtoms(head, tail)
    if bond is not None and bond.GetBondType() == Chem.BondType.DOUBLE:
        bond.SetBondType(Chem.BondType.SINGLE)
    stars = []
    for atom in (head, tail):
        star = editable.AddAtom(Chem.Atom(0))
        editable.AddBond(atom, star, Chem.BondType.SINGLE)
        stars.append(star)
    # Each linking atom gives one hydrogen to its new bond; other atoms keep theirs.
    for atom in (head, tail):
        linking = editable.GetAtomWithIdx(atom)
        if linking.GetNumExplicitHs():
            linking.SetNumExplicitHs(linking.GetNumExplicitHs() - 1)
    unit = editable.GetMol()
    try:
        Chem.SanitizeMol(unit)
    except Exception as error:
        raise ValueError(
            f"atoms {head} and {tail} cannot both take a bond to a neighbouring unit ({error}). "
            "Choose the two atoms of the polymerizable bond, as drawn above."
        ) from None
    # Start the SMILES at the head star, so the head '*' is written first.
    return Chem.MolToSmiles(unit, rootedAtAtom=stars[0])


repeat_unit = star_sited(monomer, head_atom, tail_atom)
print("Star-sited repeat unit:", repeat_unit)

from polyzymd.builders.polymer_generator import parse_star_smiles

checked = parse_star_smiles(repeat_unit, 2, "this monomer")
unit = Chem.MolFromSmiles(repeat_unit)
show(unit, highlight=[a.GetIdx() for a in unit.GetAtoms() if a.GetAtomicNum() == 0])

## 4. Choose the caps

Each chain end gets a cap: a SMILES with one `*` on the atom that bonds to the
chain. Each end can have several caps, drawn per chain with their
probabilities (each end's probabilities sum to 1; a fixed cap has probability
1). There is no default, because the caps are part of the molecule you
simulate.

- `[*][H]` caps the end with a hydrogen; `[*]C` with a methyl group.
- PolyzyMD releases before 1.3.0rc5 modelled ATRP chains as repeat units with a
  whole monomer at each end that kept its C=C double bond, bonded through its
  CH2 carbon (a modelling choice for a termination step that restores the
  double bond). The
  cell below writes that end form of your monomer: the monomer as pasted, with
  one `*` on the head atom.
- Chains from a controlled radical polymerization can also end in an
  initiator fragment and a halide; write those as SMILES with one `*` if you
  want them.

`length` counts the two ends: `length: 5` is an end, three repeat units and an
end. The cell prints a `type: polymer` entry to paste into `solutes:` of your
config. Fill in the probabilities, length and count of your study.

In [ ]:
def end_form(mol, atom):
    """Return ``mol`` as a chain end bonded through ``atom``: one '*', the C=C kept."""
    editable = Chem.RWMol(mol)
    linking = editable.GetAtomWithIdx(atom)
    if linking.GetNumExplicitHs():
        linking.SetNumExplicitHs(linking.GetNumExplicitHs() - 1)
    star = editable.AddAtom(Chem.Atom(0))
    editable.AddBond(atom, star, Chem.BondType.SINGLE)
    end = editable.GetMol()
    Chem.SanitizeMol(end)
    return Chem.MolToSmiles(end, rootedAtAtom=star)


end_cap = end_form(monomer, head_atom)  # or "[*][H]" for a hydrogen cap, "[*]C" for a methyl
parse_star_smiles(end_cap, 1, "end_groups")
print("Chain-end form:", end_cap)
end = Chem.MolFromSmiles(end_cap)
show(end, highlight=[a.GetIdx() for a in end.GetAtoms() if a.GetAtomicNum() == 0])

name = "SBMA"  # a short monomer name
residue_name = "SBM"  # three characters, not an amino-acid, water or ion name

from polyzymd.config.schema import reserved_residue_names

assert len(residue_name) <= 3, f"residue_name {residue_name!r} is longer than 3 characters"
assert residue_name.upper() not in reserved_residue_names(), (
    f"residue_name {residue_name!r} is an amino-acid, nucleic-acid, water or ion name"
)
cap = (
    f'{{label: "A", name: "{name}", residue_name: "{residue_name}", '
    f'smiles: "{end_cap}", probability: 1.0}}'
)

entry = f"""  - type: polymer
    generation_mode: "dynamic"
    type_prefix: "{name}"
    monomers:
      - label: "A"
        probability: 1.0
        name: "{name}"
        smiles: "{repeat_unit}"
        residue_name: "{residue_name}"
    end_groups:
      head:
        - {cap}
      tail:
        - {cap}
    length: 5
    count: 3
    force_field: ["openff-2.3.0.offxml"]
    charges: {{method: nagl, model: "openff-gnn-am1bcc-1.0.0.pt"}}
"""
print(entry)

## 5. Build one short chain (optional)

This builds one chain of length 3 (an end, one repeat unit and an end) with
the PolyzyMD builder, charges it with NAGL, relaxes it and checks it, as a
build does. It takes a few seconds. The files go into a temporary folder.

In [ ]:
import tempfile

from polyzymd.builders.polymer_generator import PolymerGenerator

with tempfile.TemporaryDirectory() as folder:
    generator = PolymerGenerator.from_smiles(
        [("A", name, residue_name, repeat_unit)],
        head=[{"label": "A", "name": name, "residue_name": residue_name, "smiles": end_cap}],
        tail=[{"label": "A", "name": name, "residue_name": residue_name, "smiles": end_cap}],
        cache_directory=folder,
        force_field=["openff-2.3.0.offxml"],
        type_prefix=name,
    )
    chain = generator.generate_polymer("AAA")  # head end, one repeat unit, tail end

total = sum(chain.partial_charges.m)
print(f"{chain.n_atoms} atoms, total charge {total:+.3f} e")
print("Residues:", sorted({(a.metadata['residue_number'], a.metadata['residue_name']) for a in chain.atoms}))
show(Chem.RemoveHs(chain.to_rdkit()), size=(700, 320))